# Module 2 — Autoencoder: looking inside

**Goal.** Train a small autoencoder on MNIST, inspect its intermediate computations, and test whether its two-dimensional latent space is useful for generation.

This notebook uses exactly the network from the lecture:

\[
784\;\longrightarrow\;128\;\longrightarrow\;2\;\longrightarrow\;128\;\longrightarrow\;784.
\]

The dataset contains 28 × 28 grayscale handwritten digits. We do **not** use their class labels for training: the network only learns to reproduce its inputs. We use labels later to colour a plot.

**Work through the cells in order.** Change the variables in the final experiments and record what you observe. Runs in Colab or a local Jupyter environment with PyTorch, torchvision, NumPy, and Matplotlib; downloading MNIST requires internet access on the first run.

## 1. Load the data

Each input is an image with shape `(1, 28, 28)`: one grayscale channel and 784 pixel values in `[0, 1]`. We train on a fixed subset to keep the exercise manageable. Increase `TRAIN_N` or `EPOCHS` later if you wish.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_N = 20_000
TEST_N = 3_000
BATCH_SIZE = 256
EPOCHS = 10
LR = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

transform = transforms.ToTensor()  # 0..255 -> floating-point 0..1
train_full = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_full = datasets.MNIST("./data", train=False, download=True, transform=transform)

rng = torch.Generator().manual_seed(SEED)
train_ids = torch.randperm(len(train_full), generator=rng)[:TRAIN_N]
test_ids = torch.randperm(len(test_full), generator=rng)[:TEST_N]
train_ds = Subset(train_full, train_ids.tolist())
test_ds = Subset(test_full, test_ids.tolist())
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f"Training images: {len(train_ds):,}   Test images: {len(test_ds):,}")

**Look at the input.** The network receives a vector of numbers, not an explicit instruction that the picture is a digit.

In [ ]:
images, labels = next(iter(test_loader))
image = images[0, 0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(image, cmap="gray_r", vmin=0, vmax=1)
axes[0].set_title(f"Input image (label: {labels[0].item()})")
axes[0].axis("off")
axes[1].imshow(image, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
axes[1].set_xticks(np.arange(-.5, 28, 1), minor=True)
axes[1].set_yticks(np.arange(-.5, 28, 1), minor=True)
axes[1].grid(which="minor", alpha=.2)
axes[1].tick_params(which="both", bottom=False, left=False, labelbottom=False, labelleft=False)
axes[1].set_title("28 × 28 pixel grid")
axes[2].plot(image.reshape(-1), lw=1)
axes[2].set(xlabel="Pixel index (0–783)", ylabel="Intensity", ylim=(-.04, 1.04), title="The 784 numbers")
fig.tight_layout()
plt.show()

## 2. Build the autoencoder

The **encoder** takes 784 pixel values, computes 128 hidden activations, then reduces them to two numbers \(z_1,z_2\). The **decoder** uses those two numbers to reconstruct 784 pixel values.

We use `ReLU` between linear layers and `sigmoid` at the output so each reconstructed pixel lies in `[0, 1]`. There is no activation on the two latent coordinates: they can be negative or positive.

The methods `encode` and `decode` are separate on purpose. Later, a VAE will change `encode` and the training loss, while leaving the decoder largely unchanged.

In [ ]:
class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Linear(784, 128)
        self.enc2 = nn.Linear(128, 2)
        self.dec1 = nn.Linear(2, 128)
        self.dec2 = nn.Linear(128, 784)

    def encode(self, x):
        x = x.reshape(x.size(0), 784)
        h = torch.relu(self.enc1(x))
        z = self.enc2(h)
        return z

    def decode(self, z):
        h = torch.relu(self.dec1(z))
        x_hat = torch.sigmoid(self.dec2(h))
        return x_hat.reshape(-1, 1, 28, 28)

    def forward(self, x):
        return self.decode(self.encode(x))

model = AE().to(DEVICE)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

**Check the shapes.** `batch` means that the network processes many images at once. Each image still has only two latent coordinates.

In [ ]:
batch = images[:8].to(DEVICE)
with torch.no_grad():
    flat = batch.reshape(8, 784)
    hidden = torch.relu(model.enc1(flat))
    latent = model.enc2(hidden)
    output = model.decode(latent)

for name, tensor in [("Pixels", flat), ("Encoder hidden", hidden),
                     ("Latent point", latent), ("Reconstruction", output)]:
    print(f"{name:17s}: {tuple(tensor.shape)}")
assert latent.shape == (8, 2) and output.shape == (8, 1, 28, 28)

## 3. Before training: what does the decoder output?

At this stage all network weights are random. The decoder has not learned how pixels combine into recognizable digits.

In [ ]:
fixed_x, fixed_y = next(iter(test_loader))
fixed_x = fixed_x[:8].to(DEVICE)
fixed_y = fixed_y[:8]

def show_reconstructions(source, recon, title=""):
    source = source.detach().cpu().numpy()
    recon = recon.detach().cpu().numpy()
    n = len(source)
    fig, ax = plt.subplots(2, n, figsize=(1.4*n, 3.2))
    for i in range(n):
        ax[0, i].imshow(source[i, 0], cmap="gray_r", vmin=0, vmax=1)
        ax[1, i].imshow(recon[i, 0], cmap="gray_r", vmin=0, vmax=1)
        ax[0, i].axis("off")
        ax[1, i].axis("off")
    ax[0, 0].set_ylabel("Input", fontsize=11)
    ax[1, 0].set_ylabel("Output", fontsize=11)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

with torch.no_grad():
    before_training = model(fixed_x).cpu()
show_reconstructions(fixed_x, before_training, "Before training")

## 4. Train both networks together

For each minibatch:

1. Encode each image into \(z\).
2. Decode \(z\) into a reconstructed image \(\hat x\).
3. Measure the **mean squared error** between the original and reconstructed pixels.
4. Backpropagate the error through the decoder **and** encoder, and update their weights.

\[
L_{\mathrm{AE}}=\frac{1}{784}\sum_{j=1}^{784}(x_j-\hat{x}_j)^2
\]

The objective does not tell the network where different digits should be placed in the latent space. It only asks for accurate reconstructions.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
train_losses, test_losses = [], []
snapshots = {0: before_training.numpy()}

@torch.no_grad()
def test_mse():
    model.eval()
    total = 0.0
    for x, _ in test_loader:
        x = x.to(DEVICE)
        total += criterion(model(x), x).item() * len(x)
    return total / len(test_ds)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for x, _ in train_loader:
        x = x.to(DEVICE)
        x_hat = model(x)
        loss = criterion(x_hat, x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(x)

    train_losses.append(total / len(train_ds))
    test_losses.append(test_mse())
    if epoch in {1, 3, EPOCHS // 2, EPOCHS}:
        model.eval()
        with torch.no_grad():
            snapshots[epoch] = model(fixed_x).cpu().numpy()
    print(f"Epoch {epoch:2d}/{EPOCHS}: train MSE={train_losses[-1]:.4f}   test MSE={test_losses[-1]:.4f}")

## 5. What changed during training?

First inspect the reconstruction error, then the same eight digits at different training stages. Lower MSE means closer pixel values; it does not necessarily mean that every reconstruction looks equally convincing.

In [ ]:
plt.figure(figsize=(7.5, 3.4))
plt.plot(range(1, EPOCHS+1), train_losses, marker="o", label="Train")
plt.plot(range(1, EPOCHS+1), test_losses, marker="o", label="Test")
plt.xlabel("Epoch")
plt.ylabel("Reconstruction MSE (per pixel)")
plt.title("Autoencoder learning curve")
plt.legend()
plt.grid(alpha=.2)
plt.tight_layout()
plt.show()

In [ ]:
stages = sorted(snapshots)
fig, ax = plt.subplots(len(stages)+1, len(fixed_x),
                       figsize=(11, 1.45*(len(stages)+1)))
for c in range(len(fixed_x)):
    ax[0, c].imshow(fixed_x[c, 0].cpu(), cmap="gray_r", vmin=0, vmax=1)
    ax[0, c].axis("off")
for r, epoch in enumerate(stages, start=1):
    for c in range(len(fixed_x)):
        ax[r, c].imshow(snapshots[epoch][c, 0], cmap="gray_r", vmin=0, vmax=1)
        ax[r, c].axis("off")
ax[0, 0].set_ylabel("Input", fontsize=10)
for r, epoch in enumerate(stages, start=1):
    ax[r, 0].set_ylabel(f"Epoch {epoch}", fontsize=10)
fig.suptitle("One fixed set of test images throughout training")
fig.tight_layout()
plt.show()

## 6. Look inside the encoder: 128 hidden activations

The first encoder layer produces 128 numbers from the 784 pixels. Each hidden unit has its own learned weights. Below, examine which units respond most strongly to **one** test digit, and which pixel weights those units use.

These weight images are **not** drawings of the digits or semantic features guaranteed by the architecture; they simply show the learned coefficients of the first linear layer.

In [ ]:
example_i = 0  # change to 1, 2, ... 7
x_one = fixed_x[example_i:example_i+1]
with torch.no_grad():
    h_one = torch.relu(model.enc1(x_one.reshape(1, 784)))[0].cpu().numpy()
    z_one = model.encode(x_one)[0].cpu().numpy()

top_units = np.argsort(h_one)[-4:][::-1]
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].imshow(x_one[0, 0].cpu(), cmap="gray_r", vmin=0, vmax=1)
ax[0].set_title(f"Input digit: {fixed_y[example_i].item()}")
ax[0].axis("off")
ax[1].bar(np.arange(128), h_one, width=1)
ax[1].set(xlabel="Hidden-unit index", ylabel="ReLU activation", title="128 hidden activations")
fig.tight_layout()
plt.show()
print("Latent coordinates:", np.round(z_one, 3))

weights = model.enc1.weight.detach().cpu().numpy()
fig, axes = plt.subplots(1, 4, figsize=(9, 2.4))
scale = np.max(np.abs(weights[top_units]))
for j, unit in enumerate(top_units):
    axes[j].imshow(weights[unit].reshape(28, 28), cmap="coolwarm", vmin=-scale, vmax=scale)
    axes[j].set_title(f"Unit {unit}")
    axes[j].axis("off")
fig.suptitle("Input weights for the four most active units (blue = negative; red = positive)", fontsize=11)
fig.tight_layout()
plt.show()

## 7. Look inside the bottleneck: two numbers per digit

The encoder maps every image to a point \(z=(z_1,z_2)\). Plot these points for the test set. Labels colour the points **after** training; the AE never saw those labels.

Ask: Are digits with the same label close together? Are there overlaps? Do the two coordinates have an obvious meaning?

In [ ]:
@torch.no_grad()
def encode_dataset(loader):
    model.eval()
    all_z, all_y = [], []
    for x, y in loader:
        all_z.append(model.encode(x.to(DEVICE)).cpu().numpy())
        all_y.append(y.numpy())
    return np.concatenate(all_z), np.concatenate(all_y)

z_test, y_test = encode_dataset(test_loader)
print("Latent point array:", z_test.shape)
print("First five encoded test digits (label, z1, z2):")
for label, point in zip(y_test[:5], z_test[:5]):
    print(f"  {label}: ({point[0]:+.3f}, {point[1]:+.3f})")

fig, ax = plt.subplots(figsize=(7.5, 5.8))
for digit in range(10):
    mask = y_test == digit
    ax.scatter(z_test[mask, 0], z_test[mask, 1], s=9, alpha=.55, label=str(digit))
ax.set(xlabel="z₁", ylabel="z₂", title="2D latent points of 3,000 unseen test digits")
ax.legend(title="Digit label", ncol=5, frameon=False, markerscale=2)
ax.grid(alpha=.15)
fig.tight_layout()
plt.show()

## 8. Look inside the decoder: give it points it did not receive as examples

A trained decoder accepts **any** two numbers, even ones that were never produced by the encoder. The images below come from a regular coordinate grid covering most encoded test points.

A continuous decoder gives gradually changing pixel values when we move a point. This does **not** guarantee that every decoded image is a recognizable digit.

In [ ]:
low = np.quantile(z_test, .02, axis=0)
high = np.quantile(z_test, .98, axis=0)
padding = .10 * (high - low)
low, high = low-padding, high+padding

def decode_points(points):
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(points, dtype=torch.float32, device=DEVICE)
        return model.decode(tensor).cpu().numpy()[:, 0]

N_GRID = 11
xs = np.linspace(low[0], high[0], N_GRID)
ys = np.linspace(high[1], low[1], N_GRID)  # top row has high z2
points = np.array([(x, y) for y in ys for x in xs], dtype=np.float32)
images_grid = decode_points(points).reshape(N_GRID, N_GRID, 28, 28)
# Tile the 121 decoder outputs into one large image.
mosaic = np.concatenate([np.concatenate(row, axis=1) for row in images_grid], axis=0)

fig, ax = plt.subplots(1, 2, figsize=(13, 5.4))
ax[0].scatter(z_test[:, 0], z_test[:, 1], s=4, alpha=.25)
ax[0].set(xlim=(low[0], high[0]), ylim=(low[1], high[1]),
          xlabel="z₁", ylabel="z₂", title="Encoded test points")
ax[0].grid(alpha=.15)
ax[1].imshow(mosaic, cmap="gray_r", vmin=0, vmax=1,
             extent=(low[0], high[0], low[1], high[1]), origin="upper")
ax[1].set(xlabel="z₁", ylabel="z₂", title="Decode a regular grid of coordinates")
fig.tight_layout()
plt.show()

**Test random sampling.** An ordinary AE does not require its latent points to follow a standard Gaussian. Compare decoding random encoded test points with decoding coordinates sampled from \(\mathcal N(0,I)\). The latter is an arbitrary choice for an AE: it may or may not lie in regions used during training.

In [ ]:
rng_np = np.random.default_rng(SEED)
empirical_points = z_test[rng_np.choice(len(z_test), 10, replace=False)]
gaussian_points = rng_np.normal(size=(10, 2)).astype("float32")

fig, ax = plt.subplots(2, 10, figsize=(13, 3.5))
for row, sampled in enumerate([empirical_points, gaussian_points]):
    imgs = decode_points(sampled)
    for c in range(10):
        ax[row, c].imshow(imgs[c], cmap="gray_r", vmin=0, vmax=1)
        ax[row, c].axis("off")
ax[0, 0].set_ylabel("From encoder", fontsize=10)
ax[1, 0].set_ylabel("From N(0,I)", fontsize=10)
fig.suptitle("Two ways to choose points for the same decoder")
fig.tight_layout()
plt.show()

print("Encoded test mean:", np.round(z_test.mean(axis=0), 2))
print("Encoded test std: ", np.round(z_test.std(axis=0), 2))
print("Compare with N(0,I): mean (0,0), std (1,1).")

## 9. Interpolate between two images

Take two test digits, encode them to \(z_A\) and \(z_B\), and decode points on the straight line between them:

\[
z(t)=(1-t)z_A+t z_B, \qquad 0\le t\le1.
\]

For comparison, the second row blends the **pixels** of the two images directly. The two procedures are mathematically different.

In [ ]:
def first_test_digit(digit):
    for image, label in test_ds:
        if label == digit:
            return image.unsqueeze(0).to(DEVICE)
    raise ValueError(f"No digit {digit} in the test subset")

DIGIT_A, DIGIT_B = 3, 8  # change these
x_a = first_test_digit(DIGIT_A)
x_b = first_test_digit(DIGIT_B)

with torch.no_grad():
    z_a = model.encode(x_a)[0]
    z_b = model.encode(x_b)[0]

t_values = torch.linspace(0, 1, 11, device=DEVICE)
z_path = (1 - t_values[:, None]) * z_a + t_values[:, None] * z_b
latent_images = decode_points(z_path.detach().cpu().numpy())
pixel_images = np.stack([((1-t)*x_a+t*x_b)[0, 0].cpu().numpy() for t in t_values])

fig, ax = plt.subplots(2, len(t_values), figsize=(13, 3.6))
for i, t in enumerate(t_values):
    ax[0, i].imshow(latent_images[i], cmap="gray_r", vmin=0, vmax=1)
    ax[1, i].imshow(pixel_images[i], cmap="gray_r", vmin=0, vmax=1)
    ax[0, i].set_title(f"{t.item():.1f}", fontsize=9)
    ax[0, i].axis("off")
    ax[1, i].axis("off")
ax[0, 0].set_ylabel("Latent", fontsize=10)
ax[1, 0].set_ylabel("Pixels", fontsize=10)
fig.suptitle(f"Digit {DIGIT_A} to digit {DIGIT_B}; columns show interpolation t")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(z_test[:, 0], z_test[:, 1], s=5, alpha=.13, label="Test codes")
path = z_path.detach().cpu().numpy()
ax.plot(path[:, 0], path[:, 1], "o-", color="black", lw=1.5, ms=3, label="Interpolation")
ax.scatter([path[0, 0], path[-1, 0]], [path[0, 1], path[-1, 1]],
           s=80, color=["tab:blue", "tab:orange"], zorder=3)
ax.annotate(str(DIGIT_A), path[0], xytext=(7, 7), textcoords="offset points")
ax.annotate(str(DIGIT_B), path[-1], xytext=(7, 7), textcoords="offset points")
ax.set(xlabel="z₁", ylabel="z₂", title="The interpolation is a straight line in latent coordinates")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 10. Your experiments

**A. Move an endpoint.** Change `DIGIT_A` and `DIGIT_B` (for example, `1` and `7`, or two examples of the same digit). Find one interpolation that looks reasonable and one that does not. Describe the visible changes.

**B. Sample where there are few data points.** In the latent map, find a region with almost no encoded test examples. Choose a coordinate there and run the decoder. Compare it with a coordinate in a dense region. Does the output look like a digit?

**C. Inspect a different input.** Change `example_i` in the hidden-unit section. Which hidden units become most active? Are the same units active for every digit?

**D. Answer in two or three sentences:** What has this AE learned to do well? What has it *not* been trained to do when we choose a random point \(z\)?

Try a coordinate of your own below:

In [ ]:
MY_Z = (0.0, 0.0)  # replace with coordinates from the latent plot
my_image = decode_points([MY_Z])[0]
plt.figure(figsize=(3.2, 3.2))
plt.imshow(my_image, cmap="gray_r", vmin=0, vmax=1)
plt.title(f"Decoder output at z = {MY_Z}")
plt.axis("off")
plt.show()

## 11. Save your AE and prepare for the VAE

The encoder is deterministic: for a given input it produces one fixed point \(z\). The decoder learns reconstruction, but we never specify a probability distribution from which we should sample \(z\) to generate new digits.

**Next module:** How can we alter the encoder and training objective so that sampling from a known distribution becomes part of the model? We will retain the decoder and most of this notebook.

In [ ]:
torch.save(model.state_dict(), "ae_mnist_2d.pt")
print("Saved weights to ae_mnist_2d.pt")